In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit



In [ ]:
ABS_PATH_DATA = "/home/gnone/PBL_gym_recording/F110_recordings"

name_data = "20240617-135828"
# name_data = "pp_only"
name_data = "pp_allnew"
name_data = "20240617-185045"


data_path = f"{ABS_PATH_DATA}/{name_data}/car_raw_info_{name_data}.csv"
save_path = f"{ABS_PATH_DATA}/{name_data}/"

data = pd.read_csv(data_path, header=0, sep="\t")
data.head(3)

In [ ]:
# read raceline data
raceline_path = f"{ABS_PATH_DATA}/{name_data}/traj.npy"

raceline = np.load(raceline_path)
print(f"{raceline.shape = }")
print(f"{raceline[:3, :] = }")

raceline_length = np.linalg.norm(raceline[1:, :] - raceline[:-1, :], axis=1).sum()
print(f"{raceline_length = :.3f}")

In [ ]:
# plot raceline

plt.figure(figsize=(10, 10))
plt.plot(raceline[:, 0], raceline[:, 1], 'r-')
plt.axis("equal")

plt.plot(data["pos_x"], data["pos_y"], 'b.')
plt.show()


In [ ]:
# some preprocessing

## create column with modulo on frenet s 
data["frenet_s_mod"] = data["frenet_s"].copy()
data["frenet_s_mod"] %= raceline_length

# fit a piecewise linear function to the velocity profile

## first prepare the data, by padding the velocity profile with repeated values to make it periodic
velocities = data["speed"].values
frenet_positions = data["frenet_s_mod"].values

# repeat all data points where freent is higher than half the track length
mask = frenet_positions > raceline_length / 2
frenet_positions_before = frenet_positions[mask] - raceline_length
frenet_positions = np.concatenate([frenet_positions, frenet_positions_before])
velocities = np.concatenate([velocities, velocities[mask]])
# same but after 
mask = frenet_positions < raceline_length / 2
frenet_positions_after = frenet_positions[mask] + raceline_length
frenet_positions = np.concatenate([frenet_positions, frenet_positions_after])
velocities = np.concatenate([velocities, velocities[mask]])

# TODO somehow handle the data


In [ ]:
#plot velocities against s_mod

plt.Figure(figsize=(10, 10), facecolor='white')
# plot gray with alpha 0.3
plt.plot(data["frenet_s_mod"], data["speed"], 'k.', alpha=0.3, label="speed measured")

# plot pp speed input too
plt.plot(data["frenet_s_mod"], data["speed_input_pp"], 'r.', alpha=0.3, label="speed input pp")

# plot speed_input in blue
# plt.plot(data["frenet_s_mod"], data["speed_input"], 'b.', alpha=0.3, label="speed input rl")

# plot total 
# plt.plot(data["frenet_s_mod"], data["speed_input"] + data["speed_input_pp"], 'g.', alpha=0.3, label="speed total")

# add legend outisde of plot
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

# white background
plt.gca().set_facecolor('white')


# add grid
plt.grid()

# save with background
plt.savefig(f"{save_path}/speed_against_s_mod.png", bbox_inches='tight', pad_inches=0.1, dpi=300, transparent=False, facecolor='white')


In [ ]:
# plot similar modulo staitstics but for steer

plt.Figure(figsize=(20, 10))
# plot gray with alpha 0.3
plt.plot(data["frenet_s_mod"], data["yaw_angle"], 'k.', alpha=0.3, label="steer measured")

# plot pp speed input too
plt.plot(data["frenet_s_mod"], data["steering_input_pp"], 'r.', alpha=0.3, label="steer input pp")

# plot totalsteer_input in blue
plt.plot(data["frenet_s_mod"], data["steering_input"]+data["steering_input_pp"], 'b.', alpha=0.3, label="steer input rl")

# make figure huge
# plot total
plt.savefig(f"{save_path}/steer_input.png", dpi=300)



In [ ]:
# plot things without the frenet modulo thing

## create big figure
plt.figure(figsize=(20, 10))

# plot gray with alpha 0.3
plt.plot(data["frenet_s"], data["speed"], 'k-', alpha=0.3)

# plot pp speed input too
plt.plot(data["frenet_s"], data["speed_input_pp"], 'r-', alpha=0.3)

# plot speed_input in blue
plt.plot(data["frenet_s"], data["speed_input_pp"]+data["speed_input"], 'b-', alpha=0.3)

# set aspect ratio ultrawide
plt.gca().set_aspect(20)


# savefig high resolution
plt.savefig(f"{save_path}/speeds.png", dpi=300)


In [ ]:
# make a non-causal low pass filter to smooth the speed input, with cutoff frequency at 10 Hz
def low_pass_filter(x, alpha=0.1):
    y = np.zeros_like(x)
    y[0] = x[0]
    for i in range(1, len(x)):
        y[i] = alpha * x[i] + (1 - alpha) * y[i - 1]
    return y

# apply the filter to the speed input
data["speed_input_smooth"] = low_pass_filter(data["speed_input"].values, alpha=1)

# plot the smoothed speed input
plt.figure(figsize=(20, 10))
plt.plot(data["frenet_s"], data["speed_input_pp"] + data["speed_input_smooth"], 'b-')
plt.plot(data["frenet_s"], data["speed"], 'r-')
plt.gca().set_aspect(20)
plt.savefig(f"{save_path}/speed_input_smooth.png", dpi=300)

In [ ]:
# read lap times

with open(f"{ABS_PATH_DATA}/{name_data}/lap_time_list.txt", 'r') as f:
    lap_times = f.readline()
lap_times = lap_times.strip("[]").split(",")
lap_times = [float(lap_time) for lap_time in lap_times]
print(f"avg lap time = {np.mean(lap_times):.3f} s")

print(f"Average lateral deviation: {data['frenet_d'].abs().mean():.3f} m")